# Map sequencing reads to wells

**What it does.** Read the amplicon FASTQs and count which guide RNA landed in which well.

**When to use it.** For pooled CRISPR screens, once sequencing comes back. This is what turns a plate of images into a genotype-to-phenotype table.

**What you get.** A per-well barcode count table, plus QC on read quality and consensus.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.sequencing.generate_barecode_mapping`

```
generate_barecode_mapping(settings=None)
```

Turn a folder of pooled-screen FASTQ files into per-well sgRNA count tables usable by :func:`spacr.ml.perform_regression`.

In [ ]:
from spacr.sequencing import generate_barecode_mapping

## 3. Settings and API reference

Read the descriptions here, then edit only the values in the next cell. Defaults and descriptions are generated from the installed spaCR version, so the notebook stays aligned with the API.

### [`spacr.sequencing.generate_barecode_mapping`](https://einarolafsson.github.io/spacr/api/spacr/sequencing/index.html#spacr.sequencing.generate_barecode_mapping)

- **`chunk_size`** — (int) - Number of FASTQ reads read into memory and handed to each worker batch. Larger chunks cut per-batch overhead and make the progress bar coarser but raise peak RAM per job; smaller chunks stream more gently on low-memory machines. Also sets how many reads are processed when test is True. Default 100000.
- **`column_csv`** — (path) - CSV mapping column barcodes to well names; it must have 'sequence' and 'name' columns. Reads are matched verbatim against it with no reverse-complementing, so the sequences must be in the same orientation as the reads - run barecodes_reverse_complement on the file if they are not. Unmatched reads get NA for columnID. Default the bundled spacr/resources/data/barcodes_column.csv; barcode QC (sequencing_qc) instead defaults this key to empty, where the reference is optional.
- **`comp_level`** — (int) - complevel passed to the HDF5 store, 0-9. 0 disables compression (fastest write, largest file); higher values shrink annotated_reads.h5 at increasing CPU cost, and at the top of the range saving can take longer than the barcode mapping itself. Ignored when save_h5 is False. Default 5.
- **`comp_type`** — (str) - PyTables compression library used when writing annotated_reads.h5, passed to pandas HDFStore as complib: 'zlib', 'lzo', 'bzip2' or 'blosc'. 'blosc' is far faster at similar file size, 'bzip2' is smallest but slowest. Ignored entirely when save_h5 is False. Default 'zlib'.
- **`expected_end`** — (int) - Number of bases sliced out of each read starting at offset_start relative to the target_sequence hit; this window is what the regex is matched against. It must span the whole barcode block (column + gRNA + row) or the regex stops matching and reads are dropped; shorter reads are padded with 'N'. Default 89.
- **`fill_na`** — (bool) - When a barcode does not match any entry in its reference CSV, count it under its raw sequence instead of dropping it. Off, the groupby silently discards every unmatched read, so a reference with the wrong orientation produces a small clean table rather than an obviously empty one. Turn it on to see how much of the run failed to map. Default False.
- **`grna_csv`** — (path) - CSV mapping gRNA barcode sequences to gRNA names; it must have 'sequence' and 'name' columns. Reads are matched verbatim with no reverse-complementing, so orientation must match the reads (barecodes_reverse_complement flips a file). Rows whose gRNA does not match are written as NA and dropped from the counts. Default: the bundled spacr/resources/data/grna_barcodes.csv.
- **`mode`** — (str) - Read-pairing strategy for barcode extraction: 'paired' locates target_sequence in R1 and in the reverse complement of R2 and merges them base-by-base into a quality-weighted consensus; 'single' scans one mate alone, chosen by single_direction. Paired calls barcodes more accurately but discards any read whose anchor is missing from either mate. Default 'paired'.
- **`n_jobs`** — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`offset_start`** — (int) - Bases to shift from the start of the target_sequence match to the start of the extracted window; negative values move upstream to capture a barcode preceding the anchor. The start is clamped at position 0, so an over-negative value silently shifts the reading frame and the regex stops matching. Default -8.
- **`regex`** — (str) - Regex applied with re.match to each extracted read window; it must define the named groups columnID, grna and rowID, whose captured sequences are looked up in the three barcode CSVs. Non-matching reads are silently dropped, so a wrong group name or barcode orientation yields zero counts. The default captures an 8 bp column, 20-21 bp gRNA and 8 bp row barcode.
- **`row_csv`** — (path) - CSV mapping row barcodes to well names; it must have 'sequence' and 'name' columns. Reads are matched verbatim with no reverse-complementing, so the sequences must be in the same orientation as the reads - use barecodes_reverse_complement to flip the file if needed. Unmatched reads get NA for rowID. Default: the bundled spacr/resources/data/barcodes_row.csv.
- **`save_h5`** — (bool) - Also write every annotated read (consensus sequence plus its parsed row/column/gRNA barcodes and IDs) to annotated_reads.h5. The per-well counts in unique_combinations.csv and qc.csv are written either way, so set it False unless you need read-level data; True produces a very large file and compression can dominate runtime. Default True.
- **`single_direction`** — (str) - Which mate to scan when mode is 'single': 'R1' or 'R2'. The chosen file is read as-is with no reverse-complementing, so selecting 'R2' means target_sequence and regex must be written in R2 orientation or nothing will match. Ignored when mode is 'paired'. Default 'R1'.
- **`src`** — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`target_sequence`** — (str) - Constant vector sequence used as the anchor: every read is scanned for an exact match and the barcode window is then sliced relative to that hit using offset_start and expected_end. Reads without an exact match are skipped entirely, so it must be error-free and given in the orientation of the read being scanned. Default 'TGCTGTTTCCAGCATAGCTCTTAAAC'.
- **`test`** — (bool) - In classifier training, run the held-out evaluation pass (combine with train, or use alone to score an existing model). In the sequencing barcode mapper it means something different: process only the first read chunk and print a preview, so you can sanity-check the regex and barcode CSVs in seconds. Default False.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    'chunk_size': 100000,
    'column_csv': str(__import__('pathlib').Path(next(iter(__import__('spacr').__path__))).joinpath('resources', 'data', 'barcodes_column.csv')),
    'comp_level': 5,
    'comp_type': 'zlib',
    'expected_end': 89,
    'fill_na': False,
    'grna_csv': str(__import__('pathlib').Path(next(iter(__import__('spacr').__path__))).joinpath('resources', 'data', 'barcodes_grna.csv')),
    'mode': 'paired',
    'n_jobs': None,
    'offset_start': -8,
    'regex': '^(?P<columnID>.{8})TGCTG.*TAAAC(?P<grna>.{20,21})AACTT.*AGAAG(?P<rowID>.{8}).*',
    'row_csv': str(__import__('pathlib').Path(next(iter(__import__('spacr').__path__))).joinpath('resources', 'data', 'barcodes_row.csv')),
    'save_h5': True,
    'single_direction': 'R1',
    'src': 'path',
    'target_sequence': 'TGCTGTTTCCAGCATAGCTCTTAAAC',
    'test': False,
}

In [ ]:
generate_barecode_mapping(settings)

## Where the output went

A per-well barcode count table, plus QC on read quality and consensus.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.